# Predicting Next-Iteration Eqsat Memory

Compare models that predict the next iteration's memory from the current
iteration's egraph state, rule applications, timings, and live heap.

`generate.py` records per-iteration measurements for terms in
`data/seed_terms/*/terms.json`. Every byte count is the **absolute** process
live heap, the same coordinate system `--max-memory` uses, so readings are
directly comparable to a ceiling across runs and workers.

The evaluation reports absolute next memory and memory growth, then scores the
model on the decision it actually drives: stopping a run before it breaks the
memory ceiling. Cross-validation is grouped by seed term.


In [ ]:
import altair as alt
import numpy as np
import polars as pl

import iteration_data as D
import memory_model as M
import memory_plots as MP
import plots as P

alt.theme.register("analysis", enable=True)(lambda: P.THEME)


## Load the iteration traces

Load the latest seed folder into one row per iteration. Rewrite counts are
stored in `rule_<name>` columns.

In [ ]:
SEED_DIR = D.resolve_seed_dir()

iterations = D.load_iterations(SEED_DIR)
iterations.select(
    "term_size", "iter_index", "egraph_nodes", "egraph_classes", "allocated", "n_rebuilds"
).head()

### Maximum egraph memory

Peak live heap for each term, with the median, 10th–90th percentile range,
and configured memory limit overlaid. The final stop iteration is excluded
because its heap reading is a post-run total.

In [ ]:
MP.maximum_egraph_memory(iterations, SEED_DIR)


## Build the supervised frame

Each row pairs iteration `i` with iteration `i+1` memory. Run ends without a
successor are dropped.

Transitions *into* a stop iteration are kept (`keep_stop_transitions=True`) but
flagged by `next_is_stop_iter`. A ceiling crossing **is** a stop iteration, so
dropping these would leave the ceiling analysis below with no positive examples
at all. Their heap reading is a post-run total, so they are excluded from the
growth regression and used only as crossing labels.

Derived features use only current and previous iterations.


In [ ]:
all_transitions = D.build_transitions(iterations, keep_stop_transitions=True)
# Rows with a trustworthy growth target: everything except the post-run totals.
transitions = all_transitions.filter(pl.col("next_is_stop_iter").not_())

SCALARS, RULES = D.feature_columns(transitions)
FEATURES = SCALARS + RULES
print(f"{len(SCALARS)} scalar features, {len(RULES)} rewrite-rule features")

transitions.select(
    "term_size", "iter_index", "egraph_nodes", "allocated", "next_allocated", "y_log_growth"
).head()


## Target distribution

Distribution of the log memory growth ratio.

In [ ]:
MP.growth_histogram(transitions)

## Cross-validated model comparison

Three predictors are evaluated with 5-fold `GroupKFold` by seed term:

- **naive (carry forward):** current memory
- **ridge:** log-scaled features
- **gradient boosting:** raw features

`median error ×` is the exponentiated median absolute log error.

In [ ]:
metrics, predictions = M.evaluate(transitions, FEATURES, RULES)
metrics

### Do the per-rule features help?

This ablation holds the gradient-boosting estimator and grouped CV folds constant,
changing only whether the per-rule application-count columns are present.

In [ ]:
rule_ablation = M.rule_feature_ablation(transitions, SCALARS, RULES)
rule_ablation

In [ ]:
MP.metric_bars(metrics, metric="R2")

In [ ]:
MP.predicted_vs_actual(predictions, "log memory growth ratio")

In [ ]:
MP.residual_distribution(predictions, "log memory growth ratio")

Residuals by egraph size.

In [ ]:
MP.residual_vs_size(predictions, "log memory growth ratio")

## Permutation importance

The boosted model is fitted on four group folds and evaluated on the fifth.

In [ ]:
importance = M.importances(transitions, FEATURES, RULES, target="y_log_growth")
MP.importance_bars(importance)

## Catching ceiling breaks

The model exists to stop a run before it crosses `--max-memory`, so growth
accuracy is a means, not the goal. This section scores the predictions as the
Rust hook uses them: stop when
`allocated * exp(prediction + margin) >= ceiling`.

Only the *first* predicted stop in a run counts, because the hook halts the run
there; everything after it describes a future that never happens. Crossings are
therefore counted per run, not per row.

Two boundaries are compared. **raw** trusts the prediction as-is. **conservative**
adds the safety margin, the 99th percentile of held-out residuals, which shifts
predictions up to cover the runs the model underestimates.


In [ ]:
CEILINGS = (64 << 20, 128 << 20, 256 << 20, 500 << 20)

decisions, SAFETY_MARGIN = M.ceiling_sweep(all_transitions, FEATURES, RULES, CEILINGS)
print(f"safety margin {SAFETY_MARGIN:.4f} log-growth (x{np.exp(SAFETY_MARGIN):.3f})")
decisions


`recall` is the share of crossing runs stopped in time; `precision` is the share
of stopped runs that really would have crossed. `iters_warning_*` is how many
iterations of advance notice the catches gave.


In [ ]:
MP.ceiling_decisions_chart(decisions)


### How rare is a crossing?

Crossings are a small fraction of rows, and rarer the higher the ceiling. This
is why the aggregate regression scores above look strong while the decision that
matters can still be missed: a model can fit the bulk of ordinary growth well
and still miss the tail where memory runs away.


In [ ]:
base_rates = pl.DataFrame(
    [
        {
            "ceiling_mib": c / 2**20,
            "rows_below": int(below.sum()),
            "crossings": int(cross.sum()),
            "crossing_rate_%": 100 * float(cross.sum()) / max(int(below.sum()), 1),
        }
        for c in CEILINGS
        for below, cross in [M.crossing_labels(all_transitions, c)]
    ]
)
base_rates


## History window

`build_transitions(..., window=n)` adds the previous `n - 1` scalar values and
the log deltas between consecutive lags.

Lags are computed before filtering. Missing warm-up lags use the first
iteration in the run. Rule counts include only the current iteration.

In [ ]:
sweep = M.window_sweep(iterations, windows=(1, 2, 3, 4, 6, 8))
sweep.filter(pl.col("target") == "log memory growth ratio").select(
    "window", "model", "R2", "MAE (log)", "median error x", "n_features"
).sort("model", "window")

In [ ]:
MP.window_sweep_chart(sweep, metric="R2")

## Extrapolating to larger terms

Train below a split chosen from the observed sizes and test on the largest
quarter of size levels.

In [ ]:
term_sizes = sorted(transitions["term_size"].unique().to_list())
SPLIT_SIZE = term_sizes[-max(1, len(term_sizes) // 4)]
print(f"Training below size {SPLIT_SIZE}; testing at and above it")
M.size_extrapolation(transitions, FEATURES, RULES, split_size=SPLIT_SIZE)